# Real-Time Object Detection using YOLOv5 on COCO 2017

## 1. Project and Phase Explanation

This notebook prepares Google Colab for full YOLOv5 COCO training using the exact locally verified COCO 80/10/10 image-level split.

The local VS Code workflow verifies code, preprocessing, dataset manifests, and a tiny one-epoch CPU smoke test. Colab is used for future GPU training because the local environment is CPU-only.

The full local COCO image dataset is not manually uploaded. Instead, this notebook uses a compact transfer bundle containing source code, configs, manifests, metadata, and reference checksums. Official COCO 2017 archives are downloaded directly inside Colab, then `src.prepare_coco_colab` reconstructs the exact split and YOLO labels from the transferred manifests.

The local smoke test verified the wrapper and YOLOv5 training path only. It is not a final training experiment and its metrics are not final model performance.


## 2. GPU Runtime Verification


In [ ]:
import os
import platform
import sys
import torch

print("Python:", sys.version)
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA runtime:", torch.version.cuda)
print("COLAB_RELEASE_TAG:", os.environ.get("COLAB_RELEASE_TAG"))
print("COLAB_GPU:", os.environ.get("COLAB_GPU"))
if torch.cuda.is_available():
    index = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(index)
    print("GPU name:", torch.cuda.get_device_name(index))
    print("GPU memory GiB:", round(props.total_memory / 1024**3, 2))
else:
    raise RuntimeError("CUDA is unavailable. Select Runtime > Change runtime type > GPU before continuing. Do not train on CPU in Colab.")


## 3. Google Drive Mount


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/YOLOv5_COCO_Project')
for folder in ['bundles', 'datasets', 'weights', 'runs', 'evaluations', 'exports', 'logs']:
    (DRIVE_PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)
print('Drive project root:', DRIVE_PROJECT_ROOT)


## 4. Configuration


In [ ]:
from pathlib import Path

STORAGE_MODE = "runtime"  # "runtime" uses /content/datasets; "drive" uses Google Drive dataset cache
PROJECT_ROOT = Path('/content/yolov5_project')
DATASET_ROOT = Path('/content/datasets') if STORAGE_MODE == "runtime" else DRIVE_PROJECT_ROOT / 'datasets'
OUTPUT_ROOT = DRIVE_PROJECT_ROOT / 'runs'
BUNDLE_PATH = DRIVE_PROJECT_ROOT / 'bundles' / 'yolov5_colab_bundle.zip'
BUNDLE_SHA256_PATH = DRIVE_PROJECT_ROOT / 'bundles' / 'yolov5_colab_bundle.sha256'
YOLOV5_ROOT = Path('/content/yolov5')
IMAGE_SIZE = 640
BATCH_SIZE = 16
EPOCHS = 100
WORKERS = 4
SEED = 42
DEVICE = 0
START_TRAINING = False

print('STORAGE_MODE:', STORAGE_MODE)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATASET_ROOT:', DATASET_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('START_TRAINING:', START_TRAINING)


## 5. Bundle Extraction


In [ ]:
import hashlib
import shutil
import zipfile
from pathlib import Path

if not BUNDLE_PATH.exists():
    raise FileNotFoundError(f"Upload bundle first: {BUNDLE_PATH}")
if not BUNDLE_SHA256_PATH.exists():
    raise FileNotFoundError(f"Upload checksum file first: {BUNDLE_SHA256_PATH}")

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as file:
        for chunk in iter(lambda: file.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

expected_sha = BUNDLE_SHA256_PATH.read_text().split()[0]
actual_sha = sha256_file(BUNDLE_PATH)
print('Expected bundle SHA-256:', expected_sha)
print('Actual bundle SHA-256:  ', actual_sha)
if expected_sha != actual_sha:
    raise RuntimeError('Bundle checksum mismatch. Stop and re-upload the bundle/checksum pair.')

if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE_PATH) as archive:
    bad = archive.testzip()
    if bad:
        raise RuntimeError(f'Corrupt ZIP member: {bad}')
    for info in archive.infolist():
        name = Path(info.filename)
        if name.is_absolute() or '..' in name.parts:
            raise RuntimeError(f'Unsafe bundle path: {info.filename}')
    archive.extractall(PROJECT_ROOT)
print('Bundle extracted to:', PROJECT_ROOT)


## 6. Clone YOLOv5


In [ ]:
import subprocess
import shutil
from pathlib import Path

if YOLOV5_ROOT.exists():
    shutil.rmtree(YOLOV5_ROOT)
subprocess.run(['git', 'clone', '--branch', 'v7.0', '--depth', '1', 'https://github.com/ultralytics/yolov5.git', str(YOLOV5_ROOT)], check=True)
tag = subprocess.check_output(['git', '-C', str(YOLOV5_ROOT), 'describe', '--tags', '--exact-match'], text=True).strip()
commit = subprocess.check_output(['git', '-C', str(YOLOV5_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print('YOLOv5 tag:', tag)
print('YOLOv5 commit:', commit)
if tag != 'v7.0':
    raise RuntimeError('YOLOv5 checkout is not v7.0')


## 7. Dependency Installation


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'setuptools<81', 'pycocotools', 'opencv-python', 'pandas', 'numpy', 'matplotlib', 'PyYAML', 'tqdm', 'seaborn', 'thop'], check=True)

import cv2, matplotlib, numpy, pandas, pycocotools, yaml, tqdm, setuptools
import torch
print('torch:', torch.__version__)
print('torch cuda available:', torch.cuda.is_available())
print('opencv:', cv2.__version__)
print('numpy:', numpy.__version__)
print('pandas:', pandas.__version__)
print('matplotlib:', matplotlib.__version__)
print('pyyaml:', yaml.__version__)
print('tqdm:', tqdm.__version__)
print('setuptools:', setuptools.__version__)


## 8. Official COCO Download


In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, '-m', 'src.prepare_coco_colab',
    '--workspace', str(PROJECT_ROOT),
    '--storage-root', str(DATASET_ROOT),
    '--storage-mode', STORAGE_MODE,
    '--manifests-dir', str(PROJECT_ROOT / 'data' / 'splits'),
    '--download',
]
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)


## 9. Extraction and Source Validation


In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, '-m', 'src.prepare_coco_colab',
    '--workspace', str(PROJECT_ROOT),
    '--storage-root', str(DATASET_ROOT),
    '--storage-mode', STORAGE_MODE,
    '--manifests-dir', str(PROJECT_ROOT / 'data' / 'splits'),
    '--extract', '--validate',
]
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)


## 10. Exact Split Reconstruction


In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, '-m', 'src.prepare_coco_colab',
    '--workspace', str(PROJECT_ROOT),
    '--storage-root', str(DATASET_ROOT),
    '--storage-mode', STORAGE_MODE,
    '--manifests-dir', str(PROJECT_ROOT / 'data' / 'splits'),
    '--reconstruct',
]
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)


## 11. Integrity Comparison


In [ ]:
import json
from pathlib import Path

report_path = PROJECT_ROOT / 'artifacts' / 'colab_reconstruction_report.json'
report = json.loads(report_path.read_text())
rows = report.get('stages', {}).get('integrity', {}).get('rows', [])
print(f"{'item':36} {'expected':46} {'actual':46} status")
for row in rows:
    print(f"{row['item'][:36]:36} {str(row['expected'])[:46]:46} {str(row['actual'])[:46]:46} {row['status']}")
if not rows or any(row['status'] != 'PASS' for row in rows):
    raise RuntimeError('Major Colab integrity comparison failed. Stop before training.')


## 12. Dataset YAML


In [ ]:
import yaml

DATA_YAML = DATASET_ROOT / 'coco2017' / 'coco_yolo_exact_split' / 'coco_project.yaml'
with open(DATA_YAML, 'r', encoding='utf-8') as file:
    data = yaml.safe_load(file)
print(data)
assert data['nc'] == 80
assert len(data['names']) == 80
for key in ['train', 'val', 'test']:
    resolved = Path(data['path']) / data[key]
    print(key, resolved, resolved.exists())
    if not resolved.exists():
        raise RuntimeError(f'Missing dataset path: {resolved}')


## 13. Dataloader Smoke Test


In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, '-m', 'src.prepare_coco_colab',
    '--workspace', str(PROJECT_ROOT),
    '--storage-root', str(DATASET_ROOT),
    '--storage-mode', STORAGE_MODE,
    '--manifests-dir', str(PROJECT_ROOT / 'data' / 'splits'),
    '--yolov5-root', str(YOLOV5_ROOT),
    '--imgsz', str(IMAGE_SIZE),
    '--dataloader-smoke',
]
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)
smoke = json.loads((PROJECT_ROOT / 'artifacts' / 'colab_reconstruction_report.json').read_text())['stages']['dataloader_smoke']
print(smoke)


## 14. Visual Sample Check


In [ ]:
import random
import cv2
import matplotlib.pyplot as plt
import yaml
from pathlib import Path

random.seed(SEED)
with open(DATA_YAML, 'r', encoding='utf-8') as file:
    dataset = yaml.safe_load(file)
names = {int(k): v for k, v in dataset['names'].items()}
root = Path(dataset['path'])

def show_samples(split, count=3):
    image_paths = sorted((root / 'images' / split).glob('*.jpg'))
    selected = random.sample(image_paths, min(count, len(image_paths)))
    for image_path in selected:
        label_path = root / 'labels' / split / f'{image_path.stem}.txt'
        image = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
        h, w = image.shape[:2]
        for line in label_path.read_text().splitlines():
            cls, x, y, bw, bh = line.split()
            cls = int(float(cls)); x = float(x); y = float(y); bw = float(bw); bh = float(bh)
            x1 = int((x - bw / 2) * w); y1 = int((y - bh / 2) * h)
            x2 = int((x + bw / 2) * w); y2 = int((y + bh / 2) * h)
            cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(image, names[cls], (x1, max(15, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)
        plt.figure(figsize=(10, 7))
        plt.imshow(image)
        plt.title(f'{split}: {image_path.name}')
        plt.axis('off')
        plt.show()

show_samples('train', 3)
show_samples('val', 3)


## 15. Training Configuration Preview


In [ ]:
preview = {
    'model': 'YOLOv5s first; YOLOv5m/l configs are present but disabled for the first full experiment',
    'optimizer': 'SGD',
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'image_size': IMAGE_SIZE,
    'device': DEVICE,
    'workers': WORKERS,
    'seed': SEED,
    'data_yaml': str(DATA_YAML),
    'output_dir': str(OUTPUT_ROOT / 'yolov5s'),
    'augmentation': 'YOLOv5 v7.0 default hyp.scratch-low.yaml unless explicitly changed',
    'START_TRAINING': START_TRAINING,
}
for key, value in preview.items():
    print(f'{key}: {value}')
if START_TRAINING:
    print('Training is enabled, but the guarded cells below still require explicit execution.')
else:
    print('Training remains disabled. Review all PASS checks before enabling.')


## 16. Explicit Integrity Stop


In [ ]:
print('Colab setup and dataset integrity verification completed. Review all PASS results before enabling full training.')
print('Do not continue to training cells until you intentionally set START_TRAINING = True after reviewing this notebook output.')


## 17. Full Training Cells


In [ ]:
if not START_TRAINING:
    raise RuntimeError(
        "Training is disabled. Complete and review integrity checks first."
    )

import subprocess
import sys

train_cmd = [
    sys.executable, str(YOLOV5_ROOT / 'train.py'),
    '--imgsz', str(IMAGE_SIZE),
    '--batch-size', str(BATCH_SIZE),
    '--epochs', str(EPOCHS),
    '--data', str(DATA_YAML),
    '--weights', 'yolov5s.pt',
    '--optimizer', 'SGD',
    '--device', str(DEVICE),
    '--workers', str(WORKERS),
    '--seed', str(SEED),
    '--project', str(OUTPUT_ROOT / 'yolov5s'),
    '--name', 'coco2017_yolov5s_sgd',
    '--save-period', '10',
]
print('About to run:', ' '.join(train_cmd))
subprocess.run(train_cmd, cwd=YOLOV5_ROOT, check=True)
